<a href="https://colab.research.google.com/github/Aditya-Gupta171/BBC-Data-Chatbot/blob/main/BBC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
!pip install -q langchain langchain-community chromadb sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sahilkirpekar/bbcnews-dataset")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
file_path = "/root/.cache/kagglehub/datasets/sahilkirpekar/bbcnews-dataset/versions/1/BBCNews.csv"
bbc_data = pd.read_csv(file_path)
bbc_data.columns=['News_ID','Description','Tags']
print(bbc_data.head())
print(bbc_data.info())

# Cleaning and preprocessing
bbc_data['Description'] = bbc_data['Description'].fillna('')  # Handle missing descriptions
bbc_data['Tags'] = bbc_data['Tags'].fillna('')  # Handle missing tags


2. preprocess the dataset

In [ ]:
from langchain_core.documents import Document

documents = []
for _, row in bbc_data.iterrows():
    content = row['Description']
    tags = row['Tags']
    meta = {"tags": tags, "news_id": row['News_ID']}
    documents.append(Document(page_content=content, metadata=meta))

3. Add data into chromaDB

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

hg_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
# Persist directory for ChromaDB
persist_directory = '/content/chroma_db/'

# Create ChromaDB from documents
langchain_chroma = Chroma.from_documents(
    documents=documents,
    collection_name="bbc_news",
    embedding=hg_embeddings,
    persist_directory=persist_directory
)

print("ChromaDB initialized and data persisted.")

In [ ]:
!pip install -q -U bitsandbytes accelerate transformers

4. **Retrieval** and **Generation**

Load the LLM

In [ ]:
from torch import cuda, bfloat16, float16
import transformers
from transformers import AutoTokenizer
from langchain_community.llms import HuggingFacePipeline
from time import time

# Ensure bitsandbytes is installed with the correct version
!pip install -U bitsandbytes>=0.46.1

model_id = 'HuggingFaceH4/zephyr-7b-beta'
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

# Configure model for efficient GPU memory usage
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Initialize the query pipeline
query_pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=float16,
    max_new_tokens=500,
    device_map="auto",
)

llm = HuggingFacePipeline(pipeline=query_pipeline)

Make the RAG Chain and Use RetrievalQA to build the Chain and get the Responses

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Step 1: Define a custom prompt template for better guidance
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are an expert summarizer and analyst. Using the context provided, "
        "answer the question accurately and concisely. Context: {context}\n\n"
        "Question: {question}\nAnswer:"
    )
)

# Step 2: Create the retriever with optimized parameters
retriever = langchain_chroma.as_retriever(search_kwargs={"k": 3})

# Step 3: Build an enhanced RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,  # To get source documents along with the response
    chain_type_kwargs={"prompt": custom_prompt},  # Use the custom prompt
)

# Step 4: Query the RAG model
query = "What does the dataset say about sports news?"
result = qa_chain.invoke(query)

# Step 5: Print the response and sources for traceability
response = result["result"]
sources = result["source_documents"]

print(f"Query: {query}")
print(f"Response: {response}\n")

print("Sources:")
for idx, doc in enumerate(sources, 1):
    print(f"{idx}. {doc.metadata.get('tags', 'No Tags')} - {doc.page_content[:200]}...")